# The check, then lakefed's own ingest runs

Runs first, on serverless, over one group of collections (`task_collections`: a schedule's members, or
blank for every collection with enabled rows), and does what a sweep's planning does first
(`initial_check.run_check`): each database is asked whether anything of ours changed. A database with
nothing to do gets its check stamp (`database_check`) and no ingest.

Then it starts the ingestion job itself (`launcher.py`): for each collection with work, it records the
databases with work in `check_work` and starts one run of `lakefed_ingest` with `run-now`
(`task_collection` and that list's `work_id`), the way the operations app's Run now starts it. No work,
no run: no ingest cluster starts. Inside each run nothing changes: its tables run on threads and
scheduler pools.

- **Start and end** (default): it starts the runs, watches them for `confirm_s` seconds, and ends.
  A run Databricks skips (the ingestion job at its `max_concurrent_runs`, queue off), that fails at
  once, or that no poll could read in that time (unconfirmed) fails this task, named. Each run reports its own result after that. It starts no more runs than
  the ingestion job has room for (with its queue on, all: the queue holds the rest); the rest are named
  and wait for the next cycle. With `wait` true it
  polls every run until all have ended instead (`max_running`, `poll_s`, `timeout_s`), billed for the
  whole time.
- A collection whose run is still going (from an earlier cycle) is not started again, and a collection
  whose lock a sweep holds is left to that sweep.
- **Identity**: the Jobs API calls are made as this job's run-as identity, which needs
  `CAN_MANAGE_RUN` on the ingestion job (`ingest_job_id`). Each run runs as the ingestion job's own
  run-as identity.
- **Packages**: `psycopg` and the Databricks SDK come from the job's environment
  (`resources/lakefed_ingest_check.yml`), not a `%pip` here.

In [ ]:
# Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json
from concurrent.futures import ThreadPoolExecutor

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import initial_check
import launcher
import planner

In [ ]:
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


task_collections = [c.strip() for c in widget("task_collections", "").split(",") if c.strip()]
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
# Databases checked at once (one source query each; a changed database runs its batched detection).
check_parallelism = int(widget("check_parallelism", "32"))
detection_chunk = int(widget("detection_chunk", str(ct_engine.DEFAULT_DETECTION_CHUNK)))
# The same meaning as in ingest.ipynb.
freshness_interval_s = float(widget("freshness_interval_s", "0"))
reseed_on_new_columns = widget("reseed_on_new_columns", "true").strip().lower() in ("1", "true", "yes")
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = widget("secret_scope", "lakefed_ingest")
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = widget("trust_server_certificate", "true")
os.environ["LAKEFED_SQL_SNAPSHOT_ISOLATION"] = widget("snapshot_isolation", "false")
force = {d.strip() for d in widget("force_databases", "").split(",") if d.strip()}
# The start: the ingestion job, then either watch the runs for confirm_s and end, or wait for them.
ingest_job_id = int(widget("ingest_job_id", "0"))
wait = widget("wait", "false").strip().lower() in ("1", "true", "yes")
confirm_s = float(widget("confirm_s", "30"))
max_running = int(widget("max_running", "0"))
poll_s = float(widget("poll_s", "30"))
timeout_s = float(widget("timeout_s", "0"))
# Ingestion job parameters for every run: the check's own settings above, plus the run-only ones
# below when set (blank keeps the job's default).
for name in ("parallelism", "append_batch_size", "max_seeds_per_run", "max_queries_per_source",
             "table_retries", "table_retry_budget", "spark_conf"):
    widget(name, "")
ct_engine.spark_conf_settings(dbutils.widgets.get("spark_conf"))   # a bad setting fails here, before any run starts
run_parameters = launcher.run_parameters({name: dbutils.widgets.get(name) for name in launcher.RUN_PARAMETERS})
assert ingest_job_id > 0, "ingest_job_id is required: the ingestion job to start"

job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
# This attempt of the task (its task run id, from the job's {{task.run_id}}): one start and one work
# list per collection per attempt, whatever the start call's own retries do.
launch_id = widget("launch_id", "") or job_run_id
cluster_id = _safe(lambda: spark.conf.get("spark.databricks.clusterUsageTags.clusterId"), "serverless")
store = StateStore.connect_lakebase(db_instance, db_name, reconnect=True)

# A Lakebase behind this code's schema version is migrated here, as an ingest run would.
_schema_have = store.schema_version()
if _schema_have < ss.SCHEMA_VERSION:
    with open("/Workspace" + nb_dir + "/lakebase_schema.sql") as f:
        _migrated = store.migrate(f.read())
    if _migrated:
        print(f"migrated the Lakebase schema from version {_migrated['from']} to {ss.SCHEMA_VERSION}")

if not task_collections:
    task_collections = store.enabled_collections()
lock_store = StateStore.connect_lakebase(
    db_instance, db_name, application_name=ss.lock_application_name("check", job_run_id))
print(f"checking {len(task_collections)} collection(s) (check_parallelism={check_parallelism}, "
      f"force={sorted(force) or '-'}), then starting job {ingest_job_id} for each with work (wait={wait})")

In [ ]:
# Check every collection of the group; databases are checked in parallel across all of them,
# through one shared pool.
pool = ThreadPoolExecutor(max_workers=max(1, check_parallelism), thread_name_prefix="check")


def detect_one(members):
    return planner.detect_group(
        members, current_version=None,
        database_state=lambda c: ct_engine.database_ct_state(c, spark=spark),
        detect=lambda c, items: ct_engine.detect_changes(spark, c, items, chunk_size=detection_chunk))


try:
    summary = initial_check.run_check(
        store, lock_store, task_collections, detect_one, job_run_id=job_run_id, cluster_id=cluster_id,
        existing_sinks_of=lambda configs: ct_engine.existing_sinks(spark, planner.sink_checked_configs(configs)),
        parallelism=check_parallelism, interval_s=freshness_interval_s, reseed_on_new_columns=reseed_on_new_columns,
        snapshot_isolation=ct_engine.snapshot_isolation(), force=force, map_fn=pool.map)
finally:
    pool.shutdown(wait=True)
    lock_store.close()

In [ ]:
# Record each collection's work list, then start one ingestion run per collection with work.
requests = launcher.run_requests(summary["per_collection"], parameters=run_parameters, launch_id=launch_id)
for request in requests:
    store.record_check_work(request["work_id"], request["collection"], request["work"])
summary["work_lists_pruned"] = store.prune_check_work(7)
jobs = launcher.SdkJobs()
active = jobs.active_runs(ingest_job_id) if requests else []
already = launcher.active_collections(active, work_of=store.get_check_work)
room = (launcher.capacity(jobs.max_concurrent_runs(ingest_job_id), len(active), jobs.queue_enabled(ingest_job_id))
        if requests else None)
print(f"{len(requests)} collection(s) with work: "
      + (", ".join(f"{r['collection']} ({r['databases']} db)" for r in requests) or "none, so no run starts")
      + (f"; already running: {sorted(already)}" if already else ""))
result = launcher.launch(jobs, ingest_job_id, requests, launch_id=launch_id, wait=wait, confirm_s=confirm_s,
                         max_running=max_running, poll_s=poll_s, timeout_s=timeout_s, already_running=already,
                         room=room)
summary["launch"] = result
print(json.dumps(summary, indent=2, default=str))
failed = launcher.failures(result)
if failed:
    raise RuntimeError(launcher.failure_message(failed))
dbutils.notebook.exit(json.dumps({k: summary[k] for k in summary if k != "per_collection"}, default=str))